In [2]:
from langchain_openai import OpenAIEmbeddings
from pathlib import Path
from dotenv import load_dotenv
import os
import shutil
from langchain_chroma import Chroma
from uuid import uuid4
from langchain_core.documents import Document

c:\Study\generative-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Setup path and vector store

In [3]:
project_root = Path.cwd()
project_root = project_root.parent if project_root.name == "rag" else project_root

project_root 

WindowsPath('c:/Study/generative-ai')

In [4]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable is not set. Please set it in your environment or in a .env file.")

os.getenv(f"loaded env file: {project_root / '.env'}")

In [5]:
collection_name = "my_collection"  # like table name
persistence_dir = project_root / "db"  # like database name

print(collection_name)  # like table name
print(persistence_dir)  # like database name

my_collection
c:\Study\generative-ai\db


In [6]:
if persistence_dir.exists():
    print(f"Persistence directory '{persistence_dir}' already exists.")
    shutil.rmtree(persistence_dir)
    print(f"Deleted existing persistence directory '{persistence_dir}'.")
else:
    print(f"Persistence directory '{persistence_dir}' does not exist. No need to delete.")



Persistence directory 'c:\Study\generative-ai\db' already exists.
Deleted existing persistence directory 'c:\Study\generative-ai\db'.


In [7]:
embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')

vectorstore = Chroma(
    collection_name = collection_name,
    embedding_function = embeddings,
    persist_directory = persistence_dir,
)

### 2. Utility functions

In [8]:
# utitlity function to print documents

def print_documents(title, documents):
    print(f"{title}")
    for i, doc in enumerate(documents):
        print(f"Document {i + 1}:")
        print(f"Content: {doc.page_content}")
        print(f"Metadata: {doc.metadata}")
        print("-" * 40)

### 3. Create and insert Example documents

In [9]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [10]:
documents = [
    Document(
        id = str(uuid4()),
        page_content = doc["text"],
        metadata = {
            "topic": doc["topic"],
            "doc_number": doc["doc_number"],
            }
    )
    for doc in document_examples
]

In [11]:
print_documents("Prepared Document Objects:", documents)

Prepared Document Objects:
Document 1:
Content: Artificial intelligence helps machines perform tasks that usually need human reasoning.
Metadata: {'topic': 'AI', 'doc_number': 1}
----------------------------------------
Document 2:
Content: AI systems can analyze patterns in data to support predictions and automation.
Metadata: {'topic': 'AI', 'doc_number': 2}
----------------------------------------
Document 3:
Content: Responsible AI development includes fairness, transparency, and safety checks.
Metadata: {'topic': 'AI', 'doc_number': 3}
----------------------------------------
Document 4:
Content: RAG combines retrieval with generation so the model can answer using external knowledge.
Metadata: {'topic': 'RAG', 'doc_number': 4}
----------------------------------------
Document 5:
Content: A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
Metadata: {'topic': 'RAG', 'doc_number': 5}
----------------------------------------
Document 6:


In [12]:
# insert the documents into the chroma vector store, and it will create embeddings for each document and store them in the vector store.

document_ids = vectorstore.add_documents(documents)

for id in document_ids:
    print(f"Inserted document with ID: {id}")

Inserted document with ID: 07e981da-202f-4ee9-aae9-80657a367105
Inserted document with ID: 4883b7a6-ed5c-4540-875c-4d7278747a62
Inserted document with ID: 8a54f2c9-df6e-4c32-83c5-a2ce84c9c7e7
Inserted document with ID: 60ccdf96-ed01-49fd-8f9c-2e2aa024d539
Inserted document with ID: d7f3eff2-6a07-4d0d-ab6b-33f3d6c321c4
Inserted document with ID: c7acdc75-60a8-4921-9c80-fe01752f7820
Inserted document with ID: 0dbf5975-a54a-4529-9a42-c6c4e5a43383
Inserted document with ID: dab4f2f6-2dd5-4620-9855-0ee07c83add0
Inserted document with ID: 61f02274-c7da-4e0e-b8a3-2dd110e129e7
Inserted document with ID: 7665d993-2fcf-4e80-bb27-870b56c63b4f


### 4. Read the Stored Data Back

In [13]:
# The get() method returns the low-level Chroma record structure.

vector = vectorstore.get(include=["embeddings", "metadatas", "documents"])

vector

{'ids': ['07e981da-202f-4ee9-aae9-80657a367105',
  '4883b7a6-ed5c-4540-875c-4d7278747a62',
  '8a54f2c9-df6e-4c32-83c5-a2ce84c9c7e7',
  '60ccdf96-ed01-49fd-8f9c-2e2aa024d539',
  'd7f3eff2-6a07-4d0d-ab6b-33f3d6c321c4',
  'c7acdc75-60a8-4921-9c80-fe01752f7820',
  '0dbf5975-a54a-4529-9a42-c6c4e5a43383',
  'dab4f2f6-2dd5-4620-9855-0ee07c83add0',
  '61f02274-c7da-4e0e-b8a3-2dd110e129e7',
  '7665d993-2fcf-4e80-bb27-870b56c63b4f'],
 'embeddings': array([[ 0.00492096,  0.02096558,  0.01657104, ...,  0.00069427,
         -0.01638794,  0.02790833],
        [-0.01448822, -0.00543213,  0.03034973, ..., -0.02857971,
         -0.00191593,  0.04275513],
        [ 0.02270508,  0.01531982,  0.04547119, ...,  0.02301025,
          0.00806427, -0.01585388],
        ...,
        [ 0.00801086,  0.02314758,  0.02171326, ..., -0.02072144,
         -0.0178833 ,  0.01332855],
        [ 0.00882721,  0.06268311,  0.09967041, ..., -0.01583862,
         -0.01374054,  0.0368042 ],
        [ 0.01806641,  0.08551025, 

In [14]:
print(vector["embeddings"][0])

[ 0.00492096  0.02096558  0.01657104 ...  0.00069427 -0.01638794
  0.02790833]


In [15]:
selected_ids = vector['ids'][:3]
selected_ids

['07e981da-202f-4ee9-aae9-80657a367105',
 '4883b7a6-ed5c-4540-875c-4d7278747a62',
 '8a54f2c9-df6e-4c32-83c5-a2ce84c9c7e7']

In [16]:
selected_docs = vectorstore.get_by_ids(selected_ids )
selected_docs

[Document(id='07e981da-202f-4ee9-aae9-80657a367105', metadata={'doc_number': 1, 'topic': 'AI'}, page_content='Artificial intelligence helps machines perform tasks that usually need human reasoning.'),
 Document(id='4883b7a6-ed5c-4540-875c-4d7278747a62', metadata={'topic': 'AI', 'doc_number': 2}, page_content='AI systems can analyze patterns in data to support predictions and automation.'),
 Document(id='8a54f2c9-df6e-4c32-83c5-a2ce84c9c7e7', metadata={'doc_number': 3, 'topic': 'AI'}, page_content='Responsible AI development includes fairness, transparency, and safety checks.')]

### 5. Run Similarity search

In [17]:
query = "How does RAG help an LLM answer questions using outside knowledge?"

In [18]:
search_results = vectorstore.similarity_search(query, k=3)

print(query)

search_results


How does RAG help an LLM answer questions using outside knowledge?


[Document(id='60ccdf96-ed01-49fd-8f9c-2e2aa024d539', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
 Document(id='dab4f2f6-2dd5-4620-9855-0ee07c83add0', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
 Document(id='d7f3eff2-6a07-4d0d-ab6b-33f3d6c321c4', metadata={'topic': 'RAG', 'doc_number': 5}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.')]

In [19]:
search_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print(query)

search_with_scores

How does RAG help an LLM answer questions using outside knowledge?


[(Document(id='60ccdf96-ed01-49fd-8f9c-2e2aa024d539', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.8066942691802979),
 (Document(id='dab4f2f6-2dd5-4620-9855-0ee07c83add0', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
  0.9390740990638733),
 (Document(id='d7f3eff2-6a07-4d0d-ab6b-33f3d6c321c4', metadata={'topic': 'RAG', 'doc_number': 5}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  1.0749855041503906)]

### 6. Update existing document

In [20]:
ids_to_update = [document_ids[3], document_ids[7]]  # Update the RAG documents
ids_to_update

['60ccdf96-ed01-49fd-8f9c-2e2aa024d539',
 'dab4f2f6-2dd5-4620-9855-0ee07c83add0']

In [21]:
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
Document 1:
Content: RAG improves answer quality by retrieving relevant context before the language model generates a response.
Metadata: {'topic': 'RAG', 'doc_number': 4}
----------------------------------------
Document 2:
Content: Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.
Metadata: {'topic': 'LLM', 'doc_number': 8}
----------------------------------------


In [22]:
vectorstore.update_documents(ids = ids_to_update, documents = updated_documents)

In [23]:
updated_documents = vectorstore.get_by_ids(ids_to_update)
updated_documents

[Document(id='60ccdf96-ed01-49fd-8f9c-2e2aa024d539', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG improves answer quality by retrieving relevant context before the language model generates a response.'),
 Document(id='dab4f2f6-2dd5-4620-9855-0ee07c83add0', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.')]

In [24]:
updated_query = "How can retrieved context improve an LLM response in RAG?"

In [25]:
search_result = vectorstore.similarity_search(updated_query, k=3)

search_result

[Document(id='60ccdf96-ed01-49fd-8f9c-2e2aa024d539', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG improves answer quality by retrieving relevant context before the language model generates a response.'),
 Document(id='d7f3eff2-6a07-4d0d-ab6b-33f3d6c321c4', metadata={'topic': 'RAG', 'doc_number': 5}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
 Document(id='dab4f2f6-2dd5-4620-9855-0ee07c83add0', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.')]

### 7. Delete Documents

In [26]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]

In [27]:
vectorstore.delete(ids_to_delete)

In [28]:
ids_to_delete

['61f02274-c7da-4e0e-b8a3-2dd110e129e7',
 '7665d993-2fcf-4e80-bb27-870b56c63b4f']

In [31]:
remaining_documents = vectorstore.get(include=["metadatas", "documents"])
remaining_documents

print(f"Remaining documents count: {len(remaining_documents['ids'])}")

Remaining documents count: 8


In [ ]:
remaining_documents['metadatas']

[{'doc_number': 1, 'topic': 'AI'},
 {'topic': 'AI', 'doc_number': 2},
 {'doc_number': 3, 'topic': 'AI'},
 {'topic': 'RAG', 'doc_number': 4},
 {'topic': 'RAG', 'doc_number': 5},
 {'topic': 'RAG', 'doc_number': 6},
 {'topic': 'LLM', 'doc_number': 7},
 {'doc_number': 8, 'topic': 'LLM'}]

In [42]:
print([metadata['topic'] for metadata in remaining_documents['metadatas']])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
